In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

print("Libraries imported successfully")

Libraries imported successfully


In [5]:
events = pd.read_csv("E:/Product_Analyst/Data/raw/product_app_events_raw.csv")
orders = pd.read_csv("E:/Product_Analyst/Data/raw/product_orders_raw.csv")

In [6]:
print("Events:", events.shape)
print("Orders:", orders.shape)

display(events.head())
display(orders.head())

Events: (12035, 12)
Orders: (3612, 16)


,event_id,event_timestamp,user_id,session_id,event_name,product_id,product_name,category,city,device,traffic_source,screen_name
0,EVT0000001,2023-12-18 10:00:48,U00457,S004468,view_item,P1003,Bluetooth Speaker,Electronics,hyderabad,Android,Referral,Cart
1,EVT0000002,2024-12-08 23:39:41,U02299,S004828,add_to_cart,P3001,Mixer Grinder,Home & Kitchen,MUMBAI,iOS,Organic Search,Search
2,EVT0000003,2024-05-24 03:22:36,U01379,S002167,view_item,P2003,Backpack,Fashion,DELHI,Android,Direct,Cart
3,EVT0000004,2025-11-06 21:01:14,U02575,S002371,view_item,P1002,Smart Watch,Electronics,Bangalore,Android,Paid Search,Home
4,EVT0000005,2024-12-30 09:43:13,U01456,S001402,add_to_cart,P4002,Sunscreen SPF 50,Beauty,hyderabad,Web,Direct,Search


,order_id,order_date,user_id,product_id,product_name,category,city,quantity,unit_price_inr,discount_pct,gross_amount_inr,net_amount_inr,payment_method,order_status,acquisition_channel,device
0,ORD000001,2024-03-19,U02305,P2002,Cotton T-Shirt,Fashion,Kolkata,2,603.90,5.0,1207.79,1147.40,Debit Card,Delivered,Direct,Android
1,ORD000002,2026-10-27,U00202,P3003,Bedsheet Set,Home & Kitchen,Pune,1,899.94,0.0,899.94,899.94,Net Banking,Delivered,Paid Search,Android
2,ORD000003,2024-02-26,U02316,P3001,Mixer Grinder,Home & Kitchen,HYD,1,3100.13,20.0,3100.13,2480.10,COD,Delivered,Email,Android
3,ORD000004,2026-06-09,U01746,P3003,Bedsheet Set,Home & Kitchen,Chennai,4,942.26,0.0,3769.05,3769.05,COD,Cancelled,Organic Search,Android
4,ORD000005,2024-11-10,U01447,P5002,Cricket Bat,Sports,MADRAS,2,1806.36,0.0,3612.73,3612.73,Wallet,Delivered,Social Media,iOS


In [7]:
print("========== APP EVENTS ==========")

print(events.columns.tolist())
print(events.dtypes)

print("\n========== ORDERS ==========")

print(orders.columns.tolist())
print(orders.dtypes)

========== APP EVENTS ==========
['event_id', 'event_timestamp', 'user_id', 'session_id', 'event_name', 'product_id', 'product_name', 'category', 'city', 'device', 'traffic_source', 'screen_name']
event_id           object
event_timestamp    object
user_id            object
session_id         object
event_name         object
product_id         object
product_name       object
category           object
city               object
device             object
traffic_source     object
screen_name        object
dtype: object

========== ORDERS ==========
['order_id', 'order_date', 'user_id', 'product_id', 'product_name', 'category', 'city', 'quantity', 'unit_price_inr', 'discount_pct', 'gross_amount_inr', 'net_amount_inr', 'payment_method', 'order_status', 'acquisition_channel', 'device']
order_id                object
order_date              object
user_id                 object
product_id              object
product_name            object
category                object
city                  

In [8]:
# Missing values count

print("APP EVENTS MISSING VALUES")
display(events.isnull().sum())

print("\nORDERS MISSING VALUES")
display(orders.isnull().sum())

APP EVENTS MISSING VALUES


event_id              0
event_timestamp       0
user_id               0
session_id            0
event_name            0
product_id         3584
product_name       3530
category           3530
city                155
device               91
traffic_source        0
screen_name           0
dtype: int64


ORDERS MISSING VALUES


order_id                0
order_date              0
user_id                 0
product_id              0
product_name            0
category                0
city                   33
quantity                0
unit_price_inr          0
discount_pct           24
gross_amount_inr        0
net_amount_inr          0
payment_method          0
order_status            0
acquisition_channel     0
device                  0
dtype: int64

In [9]:
events_missing = pd.DataFrame({
    "Missing_Count": events.isnull().sum(),
    "Missing_Percentage": events.isnull().mean() * 100
})

orders_missing = pd.DataFrame({
    "Missing_Count": orders.isnull().sum(),
    "Missing_Percentage": orders.isnull().mean() * 100
})

display(events_missing[events_missing["Missing_Count"] > 0])
display(orders_missing[orders_missing["Missing_Count"] > 0])

,Missing_Count,Missing_Percentage
product_id,3584,29.779809
product_name,3530,29.331118
category,3530,29.331118
city,155,1.287910
device,91,0.756128


,Missing_Count,Missing_Percentage
city,33,0.913621
discount_pct,24,0.664452


In [10]:
# Check event types where product_id is missing

missing_product_analysis = (
    events.groupby("event_name")
    .agg(
        total_events=("event_id", "count"),
        missing_product_id=("product_id", lambda x: x.isnull().sum())
    )
)

missing_product_analysis["missing_percentage"] = (
    missing_product_analysis["missing_product_id"] /
    missing_product_analysis["total_events"] * 100
).round(2)

display(missing_product_analysis)

,total_events,missing_product_id,missing_percentage
event_name,,,
ADD_TO_CART,13,0,0.00
APP_OPEN,4,4,100.00
BEGIN_CHECKOUT,8,0,0.00
PURCHASE,5,0,0.00
REMOVE_FROM_CART,3,0,0.00
SEARCH,7,7,100.00
VIEW_ITEM,19,0,0.00
add_to_cart,1863,14,0.75
app_open,2136,2136,100.00


In [11]:
display(
    events[
        events["product_id"].isnull() &
        events["product_name"].notnull()
    ].head(15)
)

,event_id,event_timestamp,user_id,session_id,event_name,product_id,product_name,category,city,device,traffic_source,screen_name
889,EVT0000890,2024-12-07 20:40:47,U02147,S000741,begin_checkout,NaN,Bluetooth Speaker,Electronics,Delhi,iOS,Paid Search,Product Detail
934,EVT0000935,2025-11-25 05:16:43,U01244,S002584,view_item,NaN,Yoga Mat,Sports,DELHI,iOS,Social Media,Cart
1317,EVT0001318,2025-03-19 13:01:30,U00463,S003338,add_to_cart,NaN,Face Serum,Beauty,Chennai,Android,Paid Search,Search
1726,EVT0001727,2023-11-24 05:01:10,U00951,S000135,add_to_cart,NaN,Smart Watch,Electronics,JAIPUR,Android,Social Media,Cart
1735,EVT0001736,2023-11-29 04:46:21,U01664,S004129,view_item,NaN,Cricket Bat,Sports,HYD,iOS,Organic Search,Home
1740,EVT0001741,2023-11-11 13:37:34,U02241,S003918,add_to_cart,NaN,Wireless Earbuds,Electronics,Chennai,iOS,Direct,Home
1954,EVT0001955,2023-12-25 02:18:34,U00990,S000901,view_item,NaN,Backpack,Fashion,JAIPUR,iOS,Organic Search,Order Confirmation
2022,EVT0002023,2026-10-04 09:29:31,U00881,S002759,begin_checkout,NaN,Yoga Mat,Sports,Vizag,iOS,Email,Product Detail
2122,EVT0002123,2025-02-13 07:14:41,U00580,S004125,begin_checkout,NaN,Backpack,Fashion,Jaipur,iOS,Paid Search,Order Confirmation
2276,EVT0002277,2025-02-24 23:53:31,U02217,S002426,add_to_cart,NaN,Wireless Earbuds,Electronics,New Delhi,iOS,Email,Checkout


In [12]:
missing_discount = orders[
    orders["discount_pct"].isnull()
]

display(
    missing_discount[
        [
            "order_id",
            "quantity",
            "unit_price_inr",
            "gross_amount_inr",
            "net_amount_inr",
            "discount_pct"
        ]
    ]
)

,order_id,quantity,unit_price_inr,gross_amount_inr,net_amount_inr,discount_pct
132,ORD000133,1,442.65,442.65,376.25,NaN
230,ORD000231,1,819.78,819.78,819.78,NaN
332,ORD000333,3,2681.69,8045.08,6436.06,NaN
455,ORD000456,1,639.36,639.36,639.36,NaN
642,ORD000643,1,730.82,730.82,694.28,NaN
786,ORD000787,1,2288.49,2288.49,2059.64,NaN
789,ORD000790,1,434.05,434.05,412.35,NaN
1166,ORD001167,1,647.13,647.13,517.70,NaN
1246,ORD001247,1,2078.92,2078.92,1871.03,NaN
1295,ORD001296,2,766.12,1532.23,1302.40,NaN


In [13]:
missing_discount = missing_discount.copy()

missing_discount["calculated_discount"] = (
    (missing_discount["gross_amount_inr"] -
     missing_discount["net_amount_inr"])
    / missing_discount["gross_amount_inr"]
) * 100

display(
    missing_discount[
        [
            "order_id",
            "gross_amount_inr",
            "net_amount_inr",
            "calculated_discount"
        ]
    ]
)

,order_id,gross_amount_inr,net_amount_inr,calculated_discount
132,ORD000133,442.65,376.25,15.000565
230,ORD000231,819.78,819.78,0.000000
332,ORD000333,8045.08,6436.06,20.000050
455,ORD000456,639.36,639.36,0.000000
642,ORD000643,730.82,694.28,4.999863
786,ORD000787,2288.49,2059.64,10.000044
789,ORD000790,434.05,412.35,4.999424
1166,ORD001167,647.13,517.70,20.000618
1246,ORD001247,2078.92,1871.03,9.999904
1295,ORD001296,1532.23,1302.40,14.999706


In [14]:
# Calculate discount percentage for all orders
orders["calculated_discount"] = (
    (orders["gross_amount_inr"] - orders["net_amount_inr"])
    / orders["gross_amount_inr"]
) * 100

# Compare original and calculated discount
orders["discount_difference"] = (
    orders["discount_pct"] - orders["calculated_discount"]
).abs()

# Check records where original discount exists
display(
    orders[
        orders["discount_pct"].notnull()
    ][
        ["order_id", "discount_pct",
         "calculated_discount", "discount_difference"]
    ].head(10)
)

,order_id,discount_pct,calculated_discount,discount_difference
0,ORD000001,5.0,5.000041,0.000041
1,ORD000002,0.0,0.000000,0.000000
2,ORD000003,20.0,20.000129,0.000129
3,ORD000004,0.0,0.000000,0.000000
4,ORD000005,0.0,0.000000,0.000000
5,ORD000006,0.0,0.000000,0.000000
6,ORD000007,0.0,0.000000,0.000000
7,ORD000008,0.0,0.000000,0.000000
8,ORD000009,15.0,15.000038,0.000038
9,ORD000010,20.0,20.000829,0.000829


In [15]:
duplicate_events = events[
    events["event_id"].duplicated(keep=False)
].sort_values("event_id")

display(duplicate_events)

,event_id,event_timestamp,user_id,session_id,event_name,product_id,product_name,category,city,device,traffic_source,screen_name
485,EVT0000486,2026-02-28 06:31:50,U01179,S000978,search,NaN,NaN,NaN,Mumbai,iOS,Email,Home
12018,EVT0000486,2026-02-28 06:31:50,U01179,S000978,search,NaN,NaN,NaN,Mumbai,iOS,Email,Home
677,EVT0000678,2024-06-09 19:57:55,U02177,S002733,view_item,P2003,Backpack,Fashion,HYD,iOS,Paid Search,Search
12001,EVT0000678,2024-06-09 19:57:55,U02177,S002733,view_item,P2003,Backpack,Fashion,HYD,iOS,Paid Search,Search
1005,EVT0001006,2023-11-18 22:48:41,U02459,S001827,add_to_cart,P1001,Wireless Earbuds,Electronics,Vizag,Android,Direct,Home
...,...,...,...,...,...,...,...,...,...,...,...,...
12020,EVT0011682,2024-02-14 18:52:20,U00312,S004063,app_open,NaN,NaN,NaN,AHD,Web,Email,Order Confirmation
12013,EVT0011806,2024-07-15 17:13:08,U00301,S002791,add_to_cart,P4001,Face Serum,Beauty,AHD,Android,Social Media,Search
11805,EVT0011806,2024-07-15 17:13:08,U00301,S002791,add_to_cart,P4001,Face Serum,Beauty,AHD,Android,Social Media,Search
12031,EVT0011989,2025-09-19 11:12:25,U02319,S003463,add_to_cart,P5002,Cricket Bat,Sports,MUMBAI,Web,Organic Search,Checkout


In [16]:
events.duplicated(keep=False)

0        False
1        False
2        False
3        False
4        False
         ...  
12030     True
12031     True
12032     True
12033     True
12034     True
Length: 12035, dtype: bool

In [17]:
print("========== DUPLICATE ANALYSIS ==========")

print("\nAPP EVENTS")
print("Exact duplicate rows:", events.duplicated().sum())
print("Duplicate event IDs:", events["event_id"].duplicated().sum())

print("\nORDERS")
print("Exact duplicate rows:", orders.duplicated().sum())
print("Duplicate order IDs:", orders["order_id"].duplicated().sum())

========== DUPLICATE ANALYSIS ==========

APP EVENTS
Exact duplicate rows: 35
Duplicate event IDs: 35

ORDERS
Exact duplicate rows: 12
Duplicate order IDs: 12


In [18]:
duplicate_orders = orders[
    orders["order_id"].duplicated(keep=False)
].sort_values("order_id")

display(duplicate_orders)

,order_id,order_date,user_id,product_id,product_name,category,city,quantity,unit_price_inr,discount_pct,gross_amount_inr,net_amount_inr,payment_method,order_status,acquisition_channel,device,calculated_discount,discount_difference
32,ORD000033,2026-07-06,U00993,P6002,Python Programming,Books,NaN,2,819.72,10.0,1639.44,1475.50,Net Banking,Delivered,Referral,Android,9.999756,2.439858e-04
3606,ORD000033,2026-07-06,U00993,P6002,Python Programming,Books,NaN,2,819.72,10.0,1639.44,1475.50,Net Banking,Delivered,Referral,Android,9.999756,2.439858e-04
3611,ORD000037,2026-01-12,U02304,P2001,Running Shoes,Fashion,AHD,1,2040.02,5.0,2040.02,1938.02,UPI,Delivered,Referral,iOS,4.999951,4.901913e-05
36,ORD000037,2026-01-12,U02304,P2001,Running Shoes,Fashion,AHD,1,2040.02,5.0,2040.02,1938.02,UPI,Delivered,Referral,iOS,4.999951,4.901913e-05
3604,ORD000363,2024-09-24,U00806,P6001,Data Analytics Handbook,Books,AHD,1,456.01,0.0,456.01,456.01,Debit Card,Delivered,Organic Search,Android,0.000000,0.000000e+00
362,ORD000363,2024-09-24,U00806,P6001,Data Analytics Handbook,Books,AHD,1,456.01,0.0,456.01,456.01,Debit Card,Delivered,Organic Search,Android,0.000000,0.000000e+00
921,ORD000922,2024-07-18,U01335,P1001,Wireless Earbuds,Electronics,Visakhapatnam,1,2100.92,5.0,2100.92,1995.87,Wallet,Processing,Email,Android,5.000190,1.903928e-04
3608,ORD000922,2024-07-18,U01335,P1001,Wireless Earbuds,Electronics,Visakhapatnam,1,2100.92,5.0,2100.92,1995.87,Wallet,Processing,Email,Android,5.000190,1.903928e-04
3601,ORD001355,2026-09-28,U01148,P6001,Data Analytics Handbook,Books,Bombay,2,555.40,15.0,1110.80,944.18,Credit Card,Processing,Direct,Android,15.000000,1.776357e-15
1354,ORD001355,2026-09-28,U01148,P6001,Data Analytics Handbook,Books,Bombay,2,555.40,15.0,1110.80,944.18,Credit Card,Processing,Direct,Android,15.000000,1.776357e-15


In [19]:
# Check whether duplicate event IDs have different records

event_id_check = (
    events.groupby("event_id")
    .nunique()
)

event_conflicts = event_id_check[
    event_id_check.gt(1).any(axis=1)
]

print("Conflicting event IDs:", len(event_conflicts))


# Check duplicate order IDs

order_id_check = (
    orders.groupby("order_id")
    .nunique()
)

order_conflicts = order_id_check[
    order_id_check.gt(1).any(axis=1)
]

print("Conflicting order IDs:", len(order_conflicts))

Conflicting event IDs: 0
Conflicting order IDs: 0


In [20]:
# Preserve original datasets
events_clean = events.copy()
orders_clean = orders.copy()

# Remove exact duplicate records
events_clean = events_clean.drop_duplicates()

orders_clean = orders_clean.drop_duplicates()

print("EVENTS BEFORE:", len(events))
print("EVENTS AFTER:", len(events_clean))

print("ORDERS BEFORE:", len(orders))
print("ORDERS AFTER:", len(orders_clean))

EVENTS BEFORE: 12035
EVENTS AFTER: 12000
ORDERS BEFORE: 3612
ORDERS AFTER: 3600
